# Entangling gate with Rydberg atoms



### Initialization cells

In [1]:
pip install qutip-qip
pip install matplotlib
pip install scipy== 1.2.1
pip install numpy

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [2]:
from quantum_algorithms import (
    H_single_qubit,
    H_two_qubit,
    time_evolution_operator,
    U1_pulse,
    U2_pulse,
    U3_pulse,
    rydberg_CZ_gate,
    computational_subspace_gate,
    Z_gate, 
    process_fidelity_manual
)


In [4]:
# Installs required Python packages. Takes about 5 minutes to run on first use.

import time, os, sys
import qutip as qt
import numpy as np

### Setting up the physical operations

Here we will set up a family of Hamiltonians that is physically realizable with Rydberg atoms. These will later be used to construct specific quantum gates, e.g. the Rydberg blockade gate discussed in lecture 17. We will be using the qutip library. Of course, instead of using qutip, you can as well use your own implementation of unitaries etc. e.g. using code from previous exercises.

In [ ]:
from qutip import *
from numpy import *

We first need two functions that return operators for single-qubit and two-qubit operations acting on a two-qubit Hilbert space spanned by the states $|g\rangle$ and $|r\rangle$. Note that you therefore do not need to write functions that act on Hilbert spaces with arbitrary qubit number, which should simplify your functions. Although Problem 1) is concerned with qubits only, we wish to extend the use of these functions in Problem 2) also to three level systems. Recall that for Rydberg gates, the computational states $|0\rangle$ and $|1\rangle$ are two states in the electronic ground state manifold, while a Rydberg state $|r\rangle$ is used for mediating interactions. What do you need to take care of in order to extend these functions to n-level systems?

Single qubit operations:
$$\hat H_j^{ab} = \frac{\Omega_j}{2}e^{i\varphi_j}|a\rangle_j\langle b|+\frac{\Omega_j}{2}e^{-i\varphi_j}|b\rangle_j\langle a|-\Delta_j|b\rangle_j\langle b| $$

Two qubit operations:
$$\hat H_{j,k}^{abcd} = \frac{V_{j,k}}{2}\left(|ab\rangle \langle cd| + |cd\rangle \langle ab|\right )$$


**(a)** Define two functions that return qutip objects representing $\hat H_j^{ab}$ and $\hat H_{j,k}^{abcd}$ as a function of the parameters $\Omega_j, \varphi_j, \Delta_j,$ and $V_{j,k}$ for $j,k \in \{1,2\}$.

**(b)** Generate the Hamiltonian in matrix form corresponding to $\hat H_1^{gr}(\Omega=1,\varphi=0,\Delta=1)$ for a two-qubit system.

**(c)** Generate the Hamiltonian in matrix form corresponding to $\hat H_{1,2}^{rrrr}(V_{1,2}=1)$ for a two-qubit system.

**For the more adventurous:** You may try to use the QuTip helper function expand_operator in order to allow operations to be applied to systems with more than 2 qubits.

**(b)** Generate the Hamiltonian in matrix form corresponding to $\hat H_1^{gr}(\Omega=1,\varphi=0,\Delta=1)$ for a two-qubit system.

In [10]:
# Single-qubit Hamiltonian H_1^{gr} with Omega=1, phi=0, Delta=1

H1 = H_single_qubit(j=1, a=0, b=1, Omega=1, phi=0, Delta=1)
print("H_1^{gr} matrix:")
print(H1.full())  # Convert to matrix

H_1^{gr} matrix:
[[ 0. +0.j  0. +0.j  0.5+0.j  0. +0.j]
 [ 0. +0.j  0. +0.j  0. +0.j  0.5+0.j]
 [ 0.5+0.j  0. +0.j -1. +0.j  0. +0.j]
 [ 0. +0.j  0.5+0.j  0. +0.j -1. +0.j]]


*(c)** Generate the Hamiltonian in matrix form corresponding to $\hat H_{1,2}^{rrrr}(V_{1,2}=1)$ for a two-qubit system.

**For the more adventurous:** You may try to use the QuTip helper function expand_operator in order to allow operations to be applied to systems with more than 2 qubits.

In [13]:
# Two-qubit Hamiltonian H_{1,2}^{rrrr} with V=1

H12 = H_two_qubit(j=1, k=2, a=1, b=1, c=1, d=1, V=1)
print("\nH_{1,2}^{rrrr} matrix:")
print(H12.full())  


H_{1,2}^{rrrr} matrix:
[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j]]


d), fully compatible with later Rydberg gate construction, and you can extend it to 3-level systems by setting dim=3 in both functions.

In [11]:
# Single-qubit Hamiltonian for a 3-level atom (|0>, |1>, |r>)

H_single_3level = H_single_qubit(j=1, a=1, b=2, Omega=1, phi=0, Delta=0, dim=3)
print(H_single_3level.full())

[[0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j]
 [0. +0.j 0. +0.j 0. +0.j 0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j]
 [0. +0.j 0. +0.j 0. +0.j 0. +0.j 0. +0.j 0.5+0.j 0. +0.j 0. +0.j 0. +0.j]]


### Problem 2: Gate operator for a multipulse sequence

We are now going to construct a pulse sequence as the product of several unitary operators, each of the following form ($\hbar=1$):

$$\hat U = \exp(-i \hat H \tau)$$

The **Rydberg blockade gate (CZ)** can be written as a three-pulse sequence

$$\hat U = \hat U_3 \hat U_2 \hat U_1$$
**note the order of operations*

with
\begin{array}
$\hat U_1 &= \exp(-i\hat H_c^{r1}\tau_1)\\
\hat U_2 &= \exp\left(-i(\hat H_t^{r1}+\hat H_{c,t}^{rrrr})\tau_2\right)\\
\hat U_3 &= \exp(-i\hat H_c^{r1}\tau_1 )
\end{array}

where $\Omega_{c,t}=\Omega$, $\Delta_{c,t}=0$, $\varphi_{c,t}=0$, and $\tau_1 = \pi/\Omega$ and $\tau_2 = 2\pi/\Omega$

We assume each atom can be treated as a three state system with the states $|0\rangle$, $|1\rangle$, and $|r\rangle$ (even though the Hamiltonian explicitely couples only $|1\rangle \leftrightarrow |r\rangle$) where $|r\rangle$ is an auxiliary Rydberg state used to mediate the interactions during the gate sequence. 

**(a)** Define the time-evolution operator $\hat U_1$ acting on the first atom within the two-atom Hilbert space. 

Show that $\hat U_1$ realizes the following truth table:
\begin{array}
$|00\rangle &\rightarrow\quad |00\rangle\\
|01\rangle &\rightarrow\quad |01\rangle \\
|10\rangle &\rightarrow-i|r0\rangle \\
|11\rangle &\rightarrow-i|r1\rangle \\
\end{array}



In [14]:
import qutip as qt
import numpy as np

# Example: spontaneous decay from |r> -> |1> for one atom
gamma = 0.01  # decay rate
c_ops = [np.sqrt(gamma) * qt.basis(3,1) * qt.basis(3,2).dag()]

# Hamiltonian for first pulse
Omega = 1
Hc = H_single_qubit(j=1, a=1, b=2, Omega=Omega, phi=0, Delta=0, dim=3)

# Initial state |10>
psi0 = qt.tensor(qt.basis(3,1), qt.basis(3,0))

# Time evolution under noise
tlist = np.linspace(0, np.pi/Omega, 100)
result = qt.mesolve(Hc, psi0, tlist, c_ops)


TypeError: incompatible dimensions[[[3, 3], [3, 3]], [[3, 3], [3, 3]]], [[[3], [3]], [[3], [3]]]

In [6]:
# Basis states
zero = qt.basis(3,0)
one = qt.basis(3,1)
r = qt.basis(3,2)

basis = {
    "|00>": qt.tensor(zero, zero),
    "|01>": qt.tensor(zero, one),
    "|10>": qt.tensor(one, zero),
    "|11>": qt.tensor(one, one)
}

U1 = U1_pulse(Omega=1)

for name, state in basis.items():
    print(f"{name} -> {U1*state}")


|00> -> Quantum object: dims=[[3, 3], [1]], shape=(9, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
|01> -> Quantum object: dims=[[3, 3], [1]], shape=(9, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
|10> -> Quantum object: dims=[[3, 3], [1]], shape=(9, 1), type='ket', dtype=Dense
Qobj data =
[[0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.-1.j]
 [0.+0.j]
 [0.+0.j]]
|11> -> Quantum object: dims=[[3, 3], [1]], shape=(9, 1), type='ket', dtype=Dense
Qobj data =
[[0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.-1.j]
 [0.+0.j]]


**(b)** Define the gate operator $\hat U$. Make sure to include the two-qubit interaction in the second pulse with $V_{c,t}\gg \Omega$  (Blockade condition). Print the gate matrix in the basis of computational states $|00\rangle$, $|01\rangle$, $|10\rangle$, $|11\rangle$

In [7]:
Omega = 1
V = 100  # Strong blockade

U_full = rydberg_CZ_gate(Omega, V)
U_comp = computational_subspace_gate(U_full)

print("Rydberg CZ gate in computational basis:")
print(U_comp)


Rydberg CZ gate in computational basis:
Quantum object: dims=[[4], [4]], shape=(4, 4), type='oper', dtype=Dense, isherm=False
Qobj data =
[[ 1.        +0.j          0.        +0.j          0.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j         -1.        +0.j          0.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j         -1.        +0.j
   0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j          0.        +0.j
  -0.99987664-0.01570614j]]


**(c)** How does this gate compare to the canonical CZ gate? Try preceeding the gate with a Z gate on both qubits. Use the built in function "process_fidelity" to calculate the normalized gate fidelity. What is the dependence on $V_{c,t}$?

In [8]:
from qutip.qip import process_fidelity
import qutip.qip.operations as ops

# Ideal CZ
U_ideal = ops.cphase(0,1,np.pi)

# Fidelity
fidelity = process_fidelity(U_comp, U_ideal)
print("Gate fidelity:", fidelity)

ImportError: cannot import name 'process_fidelity' from 'qutip_qip' (/home/f73aeabd-6de4-471d-92a9-1ba552ae6153/.local/lib/python3.11/site-packages/qutip_qip/__init__.py)

**For the more adventurous:** How would you go about including technical noise into the gate description? Does your solution depend on the temporal/spectral characteristics of the noise?